PAN Card Identification Code

In [ ]:
import cv2
from skimage.metrics import structural_similarity as ssim
from google.colab import files
import numpy as np

# Function to upload an image from Colab
def upload_image():
    uploaded = files.upload()
    for filename, content in uploaded.items():
        nparr = np.frombuffer(content, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        return img

# Function to resize an image to a smaller size
def resize_image(image, width=None, height=None, inter=cv2.INTER_AREA):
    # Initialize dimensions of the image to be resized and grab the image size
    dim = None
    (h, w) = image.shape[:2]

    # If both the width and height are None, return the original image
    if width is None and height is None:
        return image

    # Check if the width is None
    if width is None:
        # Calculate the ratio of the height and construct the dimensions
        r = height / float(h)
        dim = (int(w * r), height)

    # Otherwise, the height is None
    else:
        # Calculate the ratio of the width and construct the dimensions
        r = width / float(w)
        dim = (width, int(h * r))

    # Resize the image
    resized = cv2.resize(image, dim, interpolation=inter)

    # Return the resized image
    return resized

# Function to detect faces in an image
def detect_faces(image):
    # Load the pre-trained face detection model
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    # Convert the image to grayscale for face detection
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Detect faces in the image
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    return faces

# Define the paths to your 4 stored images
stored_images = [
    "/content/TEMP1.png",
    "/content/TEMP2.jpeg",
    "/content/TEMP3.jpeg",
    "/content/TEMP4.jpg"  # Adjust file extensions as needed
]

# Upload the user's image
user_image = upload_image()
print("User image uploaded successfully!")

# Resize the user image to a smaller size (e.g., 300x300)
user_image_resized = resize_image(user_image, width=300, height=300)

# Convert the resized user image to grayscale
user_image_gray = cv2.cvtColor(user_image_resized, cv2.COLOR_BGR2GRAY)

# Convert images to grayscale (optional, can be adjusted based on your needs)user_image_gray = cv2.cvtColor(user_image_resized, cv2.COLOR_BGR2GRAY)
stored_images_gray = []
for path in stored_images:
    img = cv2.imread(path)
    if img is not None:  # Check if the image was loaded successfully
        stored_images_gray.append(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY))
    else:
        print(f"Warning: Could not load image at path: {path}")

# Calculate SSIM scores for each stored image
ssim_scores = []
user_image_height, user_image_width = user_image_gray.shape[:2] # user_image_gray is now defined
for stored_image_gray in stored_images_gray:
    # Resize the stored image to match the dimensions of the user image
    stored_image_resized = cv2.resize(stored_image_gray, (user_image_width, user_image_height))
    # Calculate SSIM using resized images
    ssim_score = ssim(user_image_gray, stored_image_resized)
    ssim_scores.append(ssim_score)

# Print the SSIM scores
# If SSIM score is higher than 0.27, check for faces
if any(score > 0.27 for score in ssim_scores):
    # Detect faces in the user image
    faces = detect_faces(user_image)

    # If only one face is detected
    if len(faces) == 1:
        print("It looks like a pan")
    else:
        print("It does not look like a pan")

    # Draw rectangles around the detected faces
    for (x, y, w, h) in faces:
        cv2.rectangle(user_image_resized, (x, y), (x+w, y+h), (0, 255, 0), 2)

    # Print the number of detected faces
    print(f"Number of faces detected: {len(faces)}")

# If SSIM in every case is less than or equal to 0.27
elif all(score <= 0.27 for score in ssim_scores):
    print("It does not look like a pan")

# Additional processing or visualization (optional)
# You can use the SSIM scores and face detection results for further analysis, display them in a table, etc.

Saving Screenshot 2024-06-16 at 8.29.33 PM.png to Screenshot 2024-06-16 at 8.29.33 PM.png
User image uploaded successfully!
It does not look like a pan


Photo Deletion

In [ ]:
import os

def delete_image_files(folder_path):
    # Define common image file extensions
    image_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp'}

    # Iterate over all files in the folder
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)

        # Check if the file has one of the image extensions
        if os.path.isfile(file_path) and os.path.splitext(filename)[1].lower() in image_extensions:
            os.remove(file_path)
            print(f'Deleted: {file_path}')

# Example usage
folder_path = '/content/'
delete_image_files(folder_path)


Deleted: /content/Screenshot 2024-06-16 at 8.29.33 PM.png
Deleted: /content/TEMP4.jpg
Deleted: /content/TEMP3.jpeg
Deleted: /content/TEMP2.jpeg
Deleted: /content/TEMP1 (1).png
Deleted: /content/AME.png
Deleted: /content/TEMP1.png
